# J4 — Statistiques, segmentation K-Means, modèle prédictif et dashboard Olist
**Notebook corrigé** pour l’architecture Docker Spark/HDFS et les tables Silver/Gold J2–J3. Exécuter dans l’ordre après redémarrage du kernel. Les tests statistiques sont exploratoires ; aucune conclusion causale.

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import DoubleType
from delta.tables import DeltaTable

spark = SparkSession.builder.appName("J4_Stats_Segmentation_Dashboard").getOrCreate()
spark.sparkContext.setLogLevel("WARN")
spark.conf.set("spark.sql.shuffle.partitions", "16")
HDFS = "hdfs://namenode:9000"
SILVER = f"{HDFS}/lakehouse/silver/orders_enriched"
GOLD = f"{HDFS}/lakehouse/gold"
silver = spark.read.format("delta").load(SILVER)
assert silver.count() > 0, "Silver vide : terminer J2"
print("Spark", spark.version, "| Silver :", silver.count(), "lignes")
print("Spark UI :", spark.sparkContext.uiWebUrl)

:: loading settings :: url = jar:file:/opt/venv/lib/python3.15/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-4d61d9d7-195b-4374-be0e-db5af212a72e;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.1 in central
	found io.delta#delta-storage;4.4.1 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta-kernel-api;4.4.1 in central
	found

Spark 4.2.0 | Silver : 99441 lignes
Spark UI : http://spark-jupyter:4043


## 1 — Qualité des données et tests d’hypothèses
Deux tests : Mann–Whitney (délais selon la valeur des commandes) et Spearman (frais de port et délais). On limite les données ramenées sur le driver pour éviter les problèmes de mémoire et de compatibilité `toPandas()`.

In [2]:
import numpy as np
from scipy import stats

sample = (silver
    .select(
        F.expr("try_cast(order_value AS DOUBLE)").alias("order_value"),
        F.expr("try_cast(delivery_days AS DOUBLE)").alias("delivery_days"),
        F.expr("try_cast(freight_value AS DOUBLE)").alias("freight_value"))
    .dropna()
    .filter((F.col("order_value") >= 0) & (F.col("freight_value") >= 0)
            & (F.col("delivery_days") >= 0))
    .orderBy(F.rand(seed=42)).limit(15000))
rows = sample.collect()
arr = np.asarray([(r.order_value, r.delivery_days, r.freight_value) for r in rows], dtype=float)
assert len(arr) >= 20, "Pas assez de commandes valides pour les tests"
values, delays, freight = arr.T
med = np.median(values)
low, high = delays[values <= med], delays[values > med]
assert len(low) >= 2 and len(high) >= 2, "Groupes insuffisants"
u, p1 = stats.mannwhitneyu(low, high, alternative="two-sided")
rho, p2 = stats.spearmanr(freight, delays)
print(f"Échantillon : {len(arr):,} commandes ; médiane valeur : {med:.2f}")
print(f"Mann–Whitney U={u:.1f}, p={p1:.4g} ; H0 : distributions des délais identiques")
print(f"Spearman rho={rho:.4f}, p={p2:.4g} ; H0 : corrélation monotone nulle")
print("Mann–Whitney :", "rejet de H0" if p1 < .05 else "H0 non rejetée")
print("Spearman :", "rejet de H0" if np.isfinite(p2) and p2 < .05 else "H0 non rejetée")
print("Attention : significativité ≠ causalité ; tests exploratoires sur échantillon.")

Échantillon : 15,000 commandes ; médiane valeur : 105.78
Mann–Whitney U=24624574.0, p=8.716e-40 ; H0 : distributions des délais identiques
Spearman rho=0.3686, p=0 ; H0 : corrélation monotone nulle
Mann–Whitney : rejet de H0
Spearman : rejet de H0
Attention : significativité ≠ causalité ; tests exploratoires sur échantillon.


## 2 — Segmentation clients par K-Means (Spark ML)
Agrégation des comportements par client, transformation `log1p`, standardisation, K-Means. La sortie est **`customer_clusters`** afin de ne pas écraser la table `customer_segments` de J3.

In [3]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans

customer_features = (silver
    .filter(F.col("customer_unique_id").isNotNull())
    .groupBy("customer_unique_id")
    .agg(
        F.countDistinct("order_id").alias("orders_count"),
        F.avg(F.expr("try_cast(order_value AS DOUBLE)")).alias("average_order_value"),
        F.avg(F.expr("try_cast(delivery_days AS DOUBLE)")).alias("average_delivery_days"),
        F.avg(F.expr("try_cast(is_late AS DOUBLE)")).alias("late_rate"))
    .fillna({"average_order_value": 0.0, "average_delivery_days": 0.0, "late_rate": 0.0})
    .filter(F.col("orders_count") > 0))
feature_cols = ["orders_count", "average_order_value", "average_delivery_days", "late_rate"]
for col in feature_cols:
    customer_features = customer_features.withColumn(
        "log_" + col,
        F.log1p(F.greatest(F.col(col).cast("double"), F.lit(0.0))))
assembler = VectorAssembler(inputCols=["log_" + x for x in feature_cols], outputCol="raw_features", handleInvalid="skip")
scaler = StandardScaler(inputCol="raw_features", outputCol="features", withStd=True, withMean=True)
prepared = Pipeline(stages=[assembler, scaler]).fit(customer_features).transform(customer_features)
assert prepared.count() >= 3, "Pas assez de clients pour K-Means"
kmeans = KMeans(k=3, seed=42, featuresCol="features", predictionCol="cluster_id", maxIter=20)
cluster_model = kmeans.fit(prepared)
clusters = cluster_model.transform(prepared)
profiles = clusters.groupBy("cluster_id").agg(
    F.count("*").alias("customers"),
    F.expr("percentile_approx(orders_count, 0.5)").alias("median_orders"),
    F.expr("percentile_approx(average_order_value, 0.5)").alias("median_order_value"),
    F.expr("percentile_approx(late_rate, 0.5)").alias("median_late_rate"))
profiles.orderBy("cluster_id").show(truncate=False)

26/11/21 15:01:31 WARN BlockReaderFactory: I/O error constructing remote block reader for block BP-1590342591-172.26.0.2-1795008710306:blk_1073742381_1563
java.net.UnknownHostException: datanode1/<unresolved>:9866
	at org.apache.hadoop.net.NetUtils.connect(NetUtils.java:618)
	at org.apache.hadoop.hdfs.DFSClient.newConnectedPeer(DFSClient.java:3076)
	at org.apache.hadoop.hdfs.client.impl.BlockReaderFactory.nextTcpPeer(BlockReaderFactory.java:829)
	at org.apache.hadoop.hdfs.client.impl.BlockReaderFactory.getRemoteBlockReaderFromTcp(BlockReaderFactory.java:754)
	at org.apache.hadoop.hdfs.client.impl.BlockReaderFactory.build(BlockReaderFactory.java:381)
	at org.apache.hadoop.hdfs.DFSInputStream.getBlockReader(DFSInputStream.java:723)
	at org.apache.hadoop.hdfs.DFSInputStream.blockSeekTo(DFSInputStream.java:653)
	at org.apache.hadoop.hdfs.DFSInputStream.readWithStrategy(DFSInputStream.java:887)
	at org.apache.hadoop.hdfs.DFSInputStream.read(DFSInputStream.java:960)
	at org.apache.hadoop.hdf

Py4JJavaError: An error occurred while calling o253.fit.
: org.apache.spark.SparkException: [FAILED_READ_FILE.NO_HINT] Encountered error while reading file hdfs://namenode:9000/lakehouse/silver/orders_enriched/part-00000-8c4dd9c9-27f8-4b95-be1d-961308e60000-c000.snappy.parquet.  SQLSTATE: KD001
	at org.apache.spark.sql.errors.QueryExecutionErrors$.cannotReadFilesError(QueryExecutionErrors.scala:940)
	at org.apache.spark.sql.execution.datasources.v2.FileDataSourceV2$.attachFilePath(FileDataSourceV2.scala:142)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.hasNext(FileScanRDD.scala:149)
	at org.apache.spark.sql.execution.FileSourceScanExec$$anon$1.hasNext(DataSourceScanExec.scala:777)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.columnartorow_nextBatch_0$(Unknown Source)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.hashAgg_doAggregateWithKeys_0$(Unknown Source)
	at org.

In [ ]:
# Attribution de labels à partir du profil observé (pas de labels arbitraires fixes)
profile_rows = profiles.collect()
by_freq = max(profile_rows, key=lambda r: (r.median_orders or 0, r.median_order_value or 0)).cluster_id
remaining = [r for r in profile_rows if r.cluster_id != by_freq]
by_late = max(remaining, key=lambda r: (r.median_late_rate or 0, r.median_order_value or 0)).cluster_id
labels = {int(r.cluster_id): ("Clients les plus actifs" if r.cluster_id == by_freq else
         "Clients exposés aux retards" if r.cluster_id == by_late else "Clients occasionnels")
         for r in profile_rows}
label_expr = F.create_map(*[x for k,v in labels.items() for x in (F.lit(k), F.lit(v))])
customer_clusters = clusters.select("customer_unique_id", *feature_cols, "cluster_id").withColumn(
    "segment", label_expr[F.col("cluster_id")])
CLUSTERS_PATH = f"{GOLD}/customer_clusters"
(customer_clusters.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").save(CLUSTERS_PATH))
print("Clusters enregistrés :", CLUSTERS_PATH, "| clients :", spark.read.format("delta").load(CLUSTERS_PATH).count())
customer_clusters.groupBy("segment").count().show(truncate=False)

## 3 — Prédiction du retard avec Random Forest
Variables explicatives disponibles à l’achat, sans `delivery_days`, `delay_days` ni `review_score` (fuite de cible). La variable cible est évaluée seulement sur les commandes avec livraison réelle connue.

In [ ]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

model_df = (silver
    .filter(F.col("order_delivered_customer_date").isNotNull()
            & F.col("order_estimated_delivery_date").isNotNull()
            & F.col("is_late").isNotNull())
    .select(
        F.col("is_late").cast("double").alias("label"),
        F.coalesce(F.expr("try_cast(order_value AS DOUBLE)"), F.lit(0.0)).alias("order_value"),
        F.coalesce(F.expr("try_cast(freight_value AS DOUBLE)"), F.lit(0.0)).alias("freight_value"),
        F.coalesce(F.col("item_count").cast("double"), F.lit(1.0)).alias("item_count"),
        F.month("order_purchase_timestamp").cast("double").alias("purchase_month"),
        F.dayofweek("order_purchase_timestamp").cast("double").alias("purchase_weekday"),
        F.coalesce(F.col("customer_state"), F.lit("UNKNOWN")).alias("customer_state"))
    .dropna())
assert model_df.count() >= 100, "Échantillon supervisé insuffisant"
train, test = model_df.randomSplit([0.8, 0.2], seed=42)
indexer = StringIndexer(inputCol="customer_state", outputCol="state_index", handleInvalid="keep")
encoder = OneHotEncoder(inputCols=["state_index"], outputCols=["state_vec"], handleInvalid="keep")
features = VectorAssembler(inputCols=["order_value", "freight_value", "item_count",
                                        "purchase_month", "purchase_weekday", "state_vec"], outputCol="features")
rf = RandomForestClassifier(labelCol="label", featuresCol="features", numTrees=40, maxDepth=7, seed=42)
model = Pipeline(stages=[indexer, encoder, features, rf]).fit(train)
predictions = model.transform(test).cache()
auc = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC").evaluate(predictions)
accuracy = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy").evaluate(predictions)
print(f"AUC ROC = {auc:.4f} | Accuracy = {accuracy:.4f}")
predictions.groupBy("label", "prediction").count().orderBy("label", "prediction").show()
predictions.unpersist()

## 4 — Inventaire Gold et préparation Streamlit
Le dashboard doit lire uniquement les tables Delta Gold. Les chemins attendus sont ceux produits par J3 et J4.

In [ ]:
required_gold = ["category_performance", "late_orders_by_state", "monthly_delivery", "customer_segments", "customer_clusters"]
for name in required_gold:
    path = f"{GOLD}/{name}"
    try:
        df = spark.read.format("delta").load(path)
        print(f"[GOLD OK] {name}: {df.count():,} lignes")
    except Exception as exc:
        print(f"[GOLD MANQUANT] {name}: {str(exc).splitlines()[0]}")
print("J4 terminé. Dashboard Streamlit : http://localhost:8501 (si le service est démarré).")